# 03 - EDA Walkthrough (thin notebook on top of the `airline_delay` package)

**Business problem:** identify which causes, airlines, airports, routes and time windows generate the most
flight disruption, and which are controllable (see `docs/business_problem.md`).

All heavy logic lives in `src/airline_delay/` (tested). This notebook only *calls* it and shows results,
so it always runs top-to-bottom after `python scripts/run_pipeline.py`.

In [ ]:
# --- Setup: make the package importable when the notebook runs from /notebooks
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import pandas as pd
from airline_delay.config import load_config, resolve_path
from airline_delay.data.loader import load_processed
from airline_delay.analysis import metrics, insights

pd.set_option("display.float_format", "{:,.2f}".format)
cfg = load_config()

In [ ]:
# --- Load the analysis-ready dataset produced by the pipeline (cleaned + engineered)
df = load_processed(resolve_path(cfg["paths"]["processed_data"]))
print(f"{len(df):,} flights, {df.shape[1]} columns")
df.head(3)

## 1. Network health - headline KPIs

In [ ]:
# Delay = arrival >= 15 min late (DOT standard); cancelled/diverted are reported separately
kpis = metrics.network_kpis(df, cfg)
pd.Series(kpis, name="value").to_frame()

## 2. What causes delay minutes?

In [ ]:
# Cause mix among delayed flights only (cause columns are zero for on-time flights)
metrics.delay_cause_summary(df, cfg).sort_values("share_pct", ascending=False)

## 3. Airline reliability (minimum-volume filtered)

In [ ]:
tables = insights.build_tables(df, cfg)
tables["airline_kpis"].sort_values("delay_rate_pct", ascending=False)[
    ["airline_name", "flights", "delay_rate_pct", "cancel_rate_pct", "avg_arr_delay", "p90_arr_delay"]]

## 4. Airport and route bottlenecks

In [ ]:
# Airports ranked by average departure delay (only airports above the minimum flight count)
metrics.rank_worst(tables["airport_kpis"], "avg_dep_delay", 10)[
    ["origin_label", "flights", "avg_dep_delay", "delay_rate_pct", "cancel_rate_pct"]]

In [ ]:
# Routes ranked by average arrival delay (only routes above the minimum flight count)
metrics.rank_worst(tables["route_kpis"], "avg_arr_delay", 10)[
    ["route", "flights", "avg_arr_delay", "delay_rate_pct"]]

## 5. When do disruptions happen?

In [ ]:
# Delay builds through the day as aircraft rotations run late (cascading delay)
tables["hour_kpis"][["scheduled_dep_hour", "flights", "avg_dep_delay", "delay_rate_pct", "cancel_rate_pct"]]

In [ ]:
# Interactive charts: build any figure from the figure factory
from airline_delay.visualization import plots
plots.fig_hourly_delay(tables, cfg).show()

In [ ]:
plots.fig_heatmap_month_hour(df, cfg).show()

## 6. Cancellations

In [ ]:
tables["cancellation_reasons"]

## 7. Findings -> recommendations

Read the auto-generated, data-driven narrative in `reports/insights.md` and map each finding to the stakeholder
decision table in `docs/business_problem.md`.

*Interpretation caveat:* all results are descriptive associations from a 1M-row sample, not causal estimates.